<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/07-joins-deep-dive/04-join-skew-salting-aqe.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 7.4 — Join skew: see it, salt it, let AQE fix it

Skew needs manufacturing (real data files here are tiny): a 2M-row
fact where 90% of rows share ONE key. We make the skew visible with
`spark_partition_id`, fix it by hand with salting, then watch AQE do
the same job automatically.

In [ ]:
import os, sys, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("7.4-join-skew")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "false")       # manual first; AQE at the end
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")  # force shuffle joins — skew needs one
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

N_KEYS = 10_000
HOT_KEY = 0

# 90% of fact rows -> key 0; the rest spread over 10k keys
fact = spark.range(0, 2_000_000).select(
    col("id").alias("row_id"),
    F.when(F.rand(seed=7) < 0.9, F.lit(HOT_KEY))
     .otherwise((F.rand(seed=8) * N_KEYS).cast("long")).alias("key"),
    (F.rand(seed=9) * 100).alias("amount"),
)
dim = spark.range(0, N_KEYS).select(
    col("id").alias("key"),
    F.concat(F.lit("attr_"), (col("id") % 50).cast("string")).alias("attr"),
)

## Diagnostic 1: the key histogram

Five seconds, run it BEFORE any fix. Top key within ~10x of the
median = your problem is elsewhere. Ours is ~18,000x.

In [ ]:
fact.groupBy("key").count().orderBy(F.desc("count")).show(5)

## Diagnostic 2: rows per partition after hash partitioning

This is exactly what a shuffle join does to the fact table. One
partition gets ~1.8M rows; its task runs while seven sit idle.

In [ ]:
(fact.repartition(8, "key")
     .groupBy(F.spark_partition_id().alias("partition"))
     .count()
     .orderBy("partition")
     .show())

In [ ]:
# feel it: the skewed shuffle join, timed
t0 = time.time()
skewed = fact.join(dim, "key")
print("rows:", skewed.count(), f"| {time.time() - t0:.1f}s")
# in the Spark UI (spark_ui(spark) -> the count's stage): one straggler task
# with max shuffle-read >> median — the hockey stick from the lesson

## Fix by hand: salting

Fact rows get ONE random salt; dim rows get EVERY salt (8x
inflation). Join on (key, salt): the hot key now spans 8 partitions.

In [ ]:
SALT = 8

fact_s = fact.withColumn("salt", (F.rand(seed=1) * SALT).cast("int"))
dim_s = dim.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT)])))

print("dim rows:", dim.count(), "->", dim_s.count())   # the price: N_KEYS * SALT

In [ ]:
# the histogram flattens — the whole point, in one picture
(fact_s.repartition(8, "key", "salt")
       .groupBy(F.spark_partition_id().alias("partition"))
       .count()
       .orderBy("partition")
       .show())

In [ ]:
t0 = time.time()
salted = fact_s.join(dim_s, ["key", "salt"]).drop("salt")
n = salted.count()
print("rows:", n, f"| {time.time() - t0:.1f}s")
assert n == skewed.count()   # same result set — salting is a physical rewrite only

## Fix automatically: AQE skew-join handling

AQE splits any shuffle partition bigger than BOTH (factor x median)
AND the byte threshold, duplicating the matching partition on the
other side — salting, applied only where needed, with real sizes.

Honest lab note: the 256 MB default threshold never fires on our
megabytes, so we lower the dials to toy values to watch it work.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1m")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "512k")

t0 = time.time()
aqe_joined = fact.join(dim, "key")   # the ORIGINAL join — no salt code at all
print("rows:", aqe_joined.count(), f"| {time.time() - t0:.1f}s")
# UI -> SQL tab -> this query -> final plan: the SortMergeJoin reports
# "number of skewed partitions: 1" and AQEShuffleRead shows the splits

## Remedy zero, for completeness

Skew is a SHUFFLE-join disease. Our dim is 10k rows — in real life
this join should have been a broadcast and skew would never come up.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
t0 = time.time()
b = fact.join(F.broadcast(dim), "key")
print("rows:", b.count(), f"| {time.time() - t0:.1f}s   (no shuffle of fact at all)")

## Exercises

1. Rerun the salted join with SALT = 2, 8, 32 and time each. Why do
   returns diminish — what does each increment of SALT cost the dim
   side, and what's the most parallelism SALT=32 can buy on local[8]?
2. Make the hot key NULL instead of 0 (nulls in a LEFT join still
   hash to one partition). Implement the null-split fix from the
   lesson with `unionByName(..., allowMissingColumns=True)` and
   verify the row count matches the naive left join.
3. Implement PARTIAL salting: split fact into hot (key == 0) and cold
   rows, salt only the hot part, run two joins, union. Compare dim
   inflation vs full salting (8x on 1 row instead of 8x on 10k).
4. With AQE's skew dials back at defaults (factor 5, threshold 256m),
   rerun the AQE cell. Does it still split? Explain using the two-
   condition rule from the lesson.

In [ ]:
# your exercise code here